**Use this script to automatically ingest all CSV files from a specified directory into a SQLite database. Each CSV file is converted into a table named after the file (without the extension). The script also logs ingestion details and execution time for monitoring purposes.**

In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time

# LOGGING CONFIGURATION
logging.basicConfig(
    filename = "Logs/vendor_trade.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

# DATABASE ENGINE
engine = create_engine('sqlite:///vendor_trade.db')

# FUNCTIONS
"""Insert DataFrame into SQLite database as a table."""
def ingest_db(df, table_name, engine):
    df.to_sql(table_name, con = engine, if_exists = 'replace', index=False,chunksize=100000)

def load_raw_data():
    '''This function will load the CSVs as dataframe and ingest into db'''
    start = time.time()
    for file in os.listdir(r'/Data'):
                           if '.csv' in file:
                               df = pd.read_csv('/Data/' + file)
                               logging.info(f'Ingesting {file} in db')
                               ingest_db(df, file[:-4], engine)

    end = time.time()
    total_time = (end - start)/60

    logging.info('------------Ingestion Complete------------')
    logging.info(f'\nTotal TIme Taken: {total_time} minutes')

# MAIN EXECUTION
if __name__=='__main__':
    load_raw_data()